# Import Libraries

In [46]:
import pandas as pd
import numpy as np
import spacy
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import BatchNormalization, SimpleRNN, GlobalAveragePooling1D, Bidirectional, MaxPooling1D, Flatten, Layer
from tensorflow.keras.layers import Dense, LSTM, Input, Dropout, GlobalMaxPooling1D, Conv1D, GRU, Bidirectional, Lambda, Attention
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint
from tensorflow.keras.metrics import F1Score
from tensorflow.keras.optimizers import Adam
from gensim.models import Word2Vec
from transformers import BertTokenizer, TFBertModel
import tensorflow as tf

pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_colwidth', None)  # Show full content in each cell
pd.set_option('display.width', 1000)  # Set max width

# Load spaCy's English model
nlp = spacy.load('en_core_web_sm')

In [47]:
def preprocess_text(text):
    # Define interrogative words to KEEP
    interrogatives = {"what", "why", "how", "who", "where", "when", "which", "whom", "whose", "no", "not",
                    "very" ,"too" ,"too" ,"just", "if", "but", "however", "without", "like"}
    custom_stopwords = set(nlp.Defaults.stop_words)
    custom_stopwords -= interrogatives

    doc = nlp(text.lower().strip())  # Lowercase and remove whitespace
    
# Process tokens: lemmatize, filter stopwords/punct/numbers, keep interrogatives
    tokens = [
        token.lemma_ 
        for token in doc 
        if (
            (not token.is_stop or token.text in interrogatives) and  # Keep interrogatives
            not token.is_punct and token.is_alpha                                  # Remove punctuation
            # (token.is_alpha or token.like_num)                       # Keep words/numbers
        )
    ]

    return ' '.join(tokens)

In [48]:
class AttentionLayer(Layer):
    def __init__(self, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(name="att_weight", shape=(input_shape[-1], 1),
                                 initializer="normal")
        self.b = self.add_weight(name="att_bias", shape=(input_shape[1], 1),
                                 initializer="zeros")
        super().build(input_shape)

    def call(self, x):
        e = tf.keras.backend.tanh(tf.keras.backend.dot(x, self.W) + self.b)
        a = tf.keras.backend.softmax(e, axis=1)
        output = x * a
        return output

In [49]:
# Tokenize input text
# Load BERT tokenizer and model
model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)
bert_model = TFBertModel.from_pretrained(model_name)

def tokenize_texts(texts, max_len):
    encodings = tokenizer(
        texts.tolist(),
        max_length=max_len,
        truncation=True,
        padding='max_length',
        return_tensors='tf'
    )

    outputs = bert_model(encodings)

    return outputs.last_hidden_state

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.weight', 'cls.predictions.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions w

In [50]:
def make_dataset(texts, labels, tokenizer, shuffle=False, max_len= 20, batch_size = 32):
    def gen():
        for t, l in zip(texts, labels):
            enc = tokenizer(
                t,
                truncation=True,
                padding='max_length',
                max_length=max_len,
                return_tensors='tf'
            )
            yield ({'input_ids': enc['input_ids'][0], 'attention_mask': enc['attention_mask'][0]}, l)

    ds = tf.data.Dataset.from_generator(
        gen,
        output_signature=(
            {'input_ids': tf.TensorSpec(shape=(max_len,), dtype=tf.int32),
             'attention_mask': tf.TensorSpec(shape=(max_len,), dtype=tf.int32)},
            tf.TensorSpec(shape=(), dtype=tf.int32)
        )
    )
    if shuffle:
        ds = ds.shuffle(buffer_size=len(texts))
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

# Pre-Processing

### Import Data

In [ ]:
mapping = {
    'knowledge': 'knowledge',
    'remember': 'knowledge',
    'comprehension': 'comprehension',
    'understand': 'comprehension',
    'application': 'application',
    'apply': 'application',
    'analysis': 'analysis',
    'analyse': 'analysis',
    'evaluation': 'evaluation',
    'evaluate': 'evaluation',
    'synthesis': 'synthesis',
    'create': 'synthesis'
}

# Load dataset
df = pd.DataFrame()
for i in [1,2,3,4,5]:
    q_df = pd.read_csv(os.getcwd().replace('notebook' , 'dataset') + '/dataset' + str(i) + '.csv')
    q_df['dataset_id'] = i
    df = pd.concat([df , q_df], )
    
df = df.reset_index(drop=True)

# Apply preprocessing
df['label'] = df['label'].str.lower()
df['label'] = df['label'].replace(mapping)

max_len = max(len(tokenizer.encode(text, add_special_tokens=True)) for text in df['question'])
print("Max sequence length:", max_len)

Max sequence length: 95


In [ ]:
train_q, test_q = train_test_split(df['question'], )

In [ ]:
mapping = {
    'knowledge': 'knowledge',
    'remember': 'knowledge',
    'comprehension': 'comprehension',
    'understand': 'comprehension',
    'application': 'application',
    'apply': 'application',
    'analysis': 'analysis',
    'analyse': 'analysis',
    'evaluation': 'evaluation',
    'evaluate': 'evaluation',
    'synthesis': 'synthesis',
    'create': 'synthesis'
}

# Load dataset
test_df = pd.read_csv(os.getcwd().replace('notebook' , 'dataset') + '/dataset' + str(1) + '.csv')
    
test_df = test_df.reset_index(drop=True)

# Apply preprocessing
test_df['label'] = test_df['label'].str.lower()
test_df['label'] = test_df['label'].replace(mapping)

x_train = df['question']
x_test = test_df['question']

## Tokenize

### BERT

In [53]:
# Parameters
num_classes = 6

# Embedding
x_train = tokenize_texts(df['question'], max_len)
x_test = tokenize_texts(test_df['question'], max_len)

In [54]:
y_mapper = {
    'knowledge' : 0,
    'comprehension' : 1,
    'application' : 2,
    'analysis' : 3,
    'synthesis' : 4,
    'evaluation' : 5
}

y_mapped = df['label'].map(y_mapper)
y_test_mapped = test_df['label'].map(y_mapper)

y_train = to_categorical(np.asarray(y_mapped))
y_test = to_categorical(np.asarray(y_test_mapped))

# Modelling

In [55]:
callbacks = [
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1),
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)
]


## 1D CNN

In [56]:
cnn_model = Sequential([
    # Input(shape=(max_len, 768)),

    Conv1D(128, 5, activation='gelu', padding= 'same', input_shape=(max_len, 768)),
    BatchNormalization(),
    Conv1D(64, 3, activation='gelu'),
    BatchNormalization(),

    GlobalMaxPooling1D(),
    Dropout(0.3),

    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='sigmoid'),
    Dropout(0.4),

    Dense(6, activation='softmax')
])
cnn_model.compile(loss='categorical_crossentropy', 
                   optimizer=Adam(learning_rate=1e-4), 
                   metrics=['accuracy',
                            F1Score(average='macro', name='f1_macro')]
                            )

cnn_model.summary()

/opt/anaconda3/envs/yt_env/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_11 (Conv1D)              │ (None, 95, 128)        │       491,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 95, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_12 (Conv1D)              │ (None, 93, 64)         │        24,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 93, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_1          │ (None, 64)             │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_17 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_30 (Dense)                │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_18 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_31 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_19 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_32 (Dense)                │ (None, 6)              │           198 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 523,494 (2.00 MB)

 Trainable params: 523,110 (2.00 MB)

 Non-trainable params: 384 (1.50 KB)

In [57]:
cnn_history = cnn_model.fit(x_train, y_train, epochs=200, batch_size = 32, validation_split= 0.2, callbacks=callbacks)

Epoch 1/200
64/64 ━━━━━━━━━━━━━━━━━━━━ 3s 24ms/step - accuracy: 0.1558 - f1_macro: 0.1385 - loss: 2.1002 - val_accuracy: 0.1089 - val_f1_macro: 0.0576 - val_loss: 1.7369 - learning_rate: 1.0000e-04
Epoch 2/200
64/64 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - accuracy: 0.2102 - f1_macro: 0.1636 - loss: 1.9032 - val_accuracy: 0.6218 - val_f1_macro: 0.2449 - val_loss: 1.5072 - learning_rate: 1.0000e-04
Epoch 3/200
64/64 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.2741 - f1_macro: 0.2066 - loss: 1.7658 - val_accuracy: 0.6376 - val_f1_macro: 0.2824 - val_loss: 1.3652 - learning_rate: 1.0000e-04
Epoch 4/200
64/64 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - accuracy: 0.2905 - f1_macro: 0.2271 - loss: 1.7192 - val_accuracy: 0.6297 - val_f1_macro: 0.2714 - val_loss: 1.2903 - learning_rate: 1.0000e-04
Epoch 5/200
64/64 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.3487 - f1_macro: 0.2762 - loss: 1.6474 - val_accuracy: 0.6455 - val_f1_macro: 0.2962 - val_loss: 1.2365 - learning_rate: 1.0000e-04
Epoch 6/20

In [58]:
history_dict = cnn_history.history

val_acc = history_dict['val_accuracy'][-20:]
val_f1 = history_dict['val_f1_macro'][-20:]

print(f"Val Accuracy: max={np.max(val_acc):.4f}, min={np.min(val_acc):.4f}, avg={np.mean(val_acc):.4f}")
print(f"Val F1:       max={np.max(val_f1):.4f}, min={np.min(val_f1):.4f}, avg={np.mean(val_f1):.4f}")


Val Accuracy: max=0.7366, min=0.6812, avg=0.7054
Val F1:       max=0.6540, min=0.6148, avg=0.6332


In [220]:
results = cnn_model.evaluate(x_test, y_test, verbose=0)

loss, acc, f1_macro = results
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {acc * 100:.2f}%")
print(f"Test F1 Macro: {f1_macro:.4f}")


Test Loss: 1.5173
Test Accuracy: 42.17%
Test F1 Macro: 0.4092


## RNN

In [ ]:
rnn_model = Sequential([
    Input(shape=(768,)),
    SimpleRNN(256,return_sequences=True, recurrent_dropout=0.1),
    GlobalAveragePooling1D(),
    Dropout(0.2),
    Dense(64, activation='swish'),
    Dense(6, activation='softmax')
])


rnn_model.compile(loss='categorical_crossentropy', 
                   optimizer=Adam(learning_rate=1e-4), 
                   metrics=['accuracy',
                            F1Score(average='macro', name='f1_macro')]
                            )

rnn_model.summary()

Model: "sequential_27"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_30 (SimpleRNN)       │ (None, 95, 128)        │       114,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_108 (Dropout)           │ (None, 95, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_31 (SimpleRNN)       │ (None, 64)             │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_109 (Dropout)           │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_102 (Dense)               │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_110 (Dropout)           │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_103 (Dense)               │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_111 (Dropout)           │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_104 (Dense)               │ (None, 6)              │           102 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 129,878 (507.34 KB)

 Trainable params: 129,878 (507.34 KB)

 Non-trainable params: 0 (0.00 B)

In [122]:
rnn_history = rnn_model.fit(x_train, y_train, epochs=200, batch_size = 32, validation_split=0.2, callbacks=callbacks)

Epoch 1/200
64/64 ━━━━━━━━━━━━━━━━━━━━ 4s 41ms/step - accuracy: 0.1676 - f1_macro: 0.1437 - loss: 2.0002 - val_accuracy: 0.3267 - val_f1_macro: 0.1655 - val_loss: 1.7460 - learning_rate: 1.0000e-04
Epoch 2/200
64/64 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step - accuracy: 0.1865 - f1_macro: 0.1563 - loss: 1.8789 - val_accuracy: 0.3901 - val_f1_macro: 0.1437 - val_loss: 1.7494 - learning_rate: 1.0000e-04
Epoch 3/200
64/64 ━━━━━━━━━━━━━━━━━━━━ 2s 36ms/step - accuracy: 0.2099 - f1_macro: 0.1769 - loss: 1.8533 - val_accuracy: 0.5129 - val_f1_macro: 0.1305 - val_loss: 1.7458 - learning_rate: 1.0000e-04
Epoch 4/200
64/64 ━━━━━━━━━━━━━━━━━━━━ 2s 36ms/step - accuracy: 0.2075 - f1_macro: 0.1529 - loss: 1.8344 - val_accuracy: 0.5723 - val_f1_macro: 0.1275 - val_loss: 1.7290 - learning_rate: 1.0000e-04
Epoch 5/200
64/64 ━━━━━━━━━━━━━━━━━━━━ 2s 36ms/step - accuracy: 0.2220 - f1_macro: 0.1677 - loss: 1.7836 - val_accuracy: 0.5921 - val_f1_macro: 0.1246 - val_loss: 1.7119 - learning_rate: 1.0000e-04
Epoch 6/20

In [ ]:
history_dict = rnn_history.history

val_acc = history_dict['val_accuracy'][-20:]
val_f1 = history_dict['val_f1_macro'][-20:]

print(f"Val Accuracy: max={np.max(val_acc):.4f}, min={np.min(val_acc):.4f}, avg={np.mean(val_acc):.4f}")
print(f"Val F1:       max={np.max(val_f1):.4f}, min={np.min(val_f1):.4f}, avg={np.mean(val_f1):.4f}")

Val Accuracy: max=0.6614, min=0.4653, avg=0.6141
Val F1:       max=0.3923, min=0.2641, avg=0.3438


In [224]:
results = rnn_model.evaluate(x_test, y_test, verbose=0)

loss, acc, f1_macro = results
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {acc * 100:.2f}%")
print(f"Test F1 Macro: {f1_macro:.4f}")

Test Loss: 1.6771
Test Accuracy: 31.67%
Test F1 Macro: 0.2395


## LSTM

In [135]:
lstm_model = Sequential([
    Input(shape=(max_len, 768)),
    LSTM(128, return_sequences=True, recurrent_dropout=0.1),
    GlobalAveragePooling1D(),
    Dropout(0.2),
    Dense(64, activation='swish'),
    Dense(6, activation='softmax')
])

lstm_model.compile(loss='categorical_crossentropy', 
                   optimizer=Adam(learning_rate=1e-4), 
                   metrics=['accuracy',
                            F1Score(average='macro', name='f1_macro')]
                            )

lstm_model.summary()

Model: "sequential_30"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_13 (LSTM)                  │ (None, 95, 128)        │       459,264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_11     │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_114 (Dropout)           │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_109 (Dense)               │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_110 (Dense)               │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 467,910 (1.78 MB)

 Trainable params: 467,910 (1.78 MB)

 Non-trainable params: 0 (0.00 B)

In [136]:
lstm_history = lstm_model.fit(x_train, y_train, epochs=200, batch_size = 32, validation_split=0.2, callbacks=callbacks)

Epoch 1/200
64/64 ━━━━━━━━━━━━━━━━━━━━ 8s 117ms/step - accuracy: 0.2839 - f1_macro: 0.1174 - loss: 1.7184 - val_accuracy: 0.6000 - val_f1_macro: 0.1250 - val_loss: 1.3474 - learning_rate: 1.0000e-04
Epoch 2/200
64/64 ━━━━━━━━━━━━━━━━━━━━ 8s 117ms/step - accuracy: 0.4105 - f1_macro: 0.2395 - loss: 1.5092 - val_accuracy: 0.6792 - val_f1_macro: 0.4056 - val_loss: 1.1675 - learning_rate: 1.0000e-04
Epoch 3/200
64/64 ━━━━━━━━━━━━━━━━━━━━ 7s 115ms/step - accuracy: 0.5222 - f1_macro: 0.4665 - loss: 1.3250 - val_accuracy: 0.7149 - val_f1_macro: 0.5589 - val_loss: 0.9874 - learning_rate: 1.0000e-04
Epoch 4/200
64/64 ━━━━━━━━━━━━━━━━━━━━ 7s 113ms/step - accuracy: 0.6158 - f1_macro: 0.5832 - loss: 1.1110 - val_accuracy: 0.7366 - val_f1_macro: 0.5873 - val_loss: 0.8781 - learning_rate: 1.0000e-04
Epoch 5/200
64/64 ━━━━━━━━━━━━━━━━━━━━ 7s 113ms/step - accuracy: 0.6354 - f1_macro: 0.6129 - loss: 1.0010 - val_accuracy: 0.7188 - val_f1_macro: 0.6180 - val_loss: 0.8637 - learning_rate: 1.0000e-04
Epoch

In [138]:
history_dict = lstm_history.history

val_acc = history_dict['val_accuracy'][-10:]
val_f1 = history_dict['val_f1_macro'][-10:]

print(f"Val Accuracy: max={np.max(val_acc):.4f}, min={np.min(val_acc):.4f}, avg={np.mean(val_acc):.4f}")
print(f"Val F1:       max={np.max(val_f1):.4f}, min={np.min(val_f1):.4f}, avg={np.mean(val_f1):.4f}")


Val Accuracy: max=0.7267, min=0.6653, avg=0.6927
Val F1:       max=0.6293, min=0.5946, avg=0.6102


In [228]:
results = lstm_model.evaluate(x_test, y_test, verbose=0)

loss, acc, f1_macro = results
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {acc * 100:.2f}%")
print(f"Test F1 Macro: {f1_macro:.4f}")

Test Loss: 1.6803
Test Accuracy: 41.33%
Test F1 Macro: 0.4100


### Bi-GRU

In [140]:
gru_model = Sequential([
    Input(shape=(max_len, 768)),
    Bidirectional(GRU(128, return_sequences=True, recurrent_dropout=0.1)),
    GlobalAveragePooling1D(),
    Dropout(0.2),
    Dense(64, activation='swish'),
    Dense(6, activation='softmax')
])

gru_model.compile(loss='categorical_crossentropy', 
                   optimizer=Adam(learning_rate=1e-4), 
                   metrics=['accuracy',
                            F1Score(average='macro', name='f1_macro')]
                            )

gru_model.summary()


Model: "sequential_32"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional_23                │ (None, 95, 256)        │       689,664 │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_13     │ (None, 256)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_116 (Dropout)           │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_113 (Dense)               │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_114 (Dense)               │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 706,502 (2.70 MB)

 Trainable params: 706,502 (2.70 MB)

 Non-trainable params: 0 (0.00 B)

In [141]:
gru_history = gru_model.fit(x_train, y_train, epochs=200, batch_size = 32, validation_split=0.2, callbacks=callbacks)

Epoch 1/200
64/64 ━━━━━━━━━━━━━━━━━━━━ 14s 193ms/step - accuracy: 0.2876 - f1_macro: 0.1288 - loss: 1.7124 - val_accuracy: 0.6139 - val_f1_macro: 0.1794 - val_loss: 1.2914 - learning_rate: 1.0000e-04
Epoch 2/200
64/64 ━━━━━━━━━━━━━━━━━━━━ 14s 222ms/step - accuracy: 0.4179 - f1_macro: 0.2610 - loss: 1.4713 - val_accuracy: 0.6455 - val_f1_macro: 0.3003 - val_loss: 1.1053 - learning_rate: 1.0000e-04
Epoch 3/200
64/64 ━━━━━━━━━━━━━━━━━━━━ 16s 256ms/step - accuracy: 0.5296 - f1_macro: 0.4505 - loss: 1.3068 - val_accuracy: 0.7109 - val_f1_macro: 0.5414 - val_loss: 1.0464 - learning_rate: 1.0000e-04
Epoch 4/200
64/64 ━━━━━━━━━━━━━━━━━━━━ 16s 253ms/step - accuracy: 0.6030 - f1_macro: 0.5600 - loss: 1.1532 - val_accuracy: 0.7248 - val_f1_macro: 0.5852 - val_loss: 0.9698 - learning_rate: 1.0000e-04
Epoch 5/200
64/64 ━━━━━━━━━━━━━━━━━━━━ 16s 256ms/step - accuracy: 0.6175 - f1_macro: 0.5807 - loss: 1.0390 - val_accuracy: 0.7347 - val_f1_macro: 0.6022 - val_loss: 0.9178 - learning_rate: 1.0000e-04


In [ ]:
history_dict = gru_history.history

val_acc = history_dict['val_accuracy'][-10:]
val_f1 = history_dict['val_f1_macro'][-10:]

print(f"Val Accuracy: max={np.max(val_acc):.4f}, min={np.min(val_acc):.4f}, avg={np.mean(val_acc):.4f}")
print(f"Val F1:       max={np.max(val_f1):.4f}, min={np.min(val_f1):.4f}, avg={np.mean(val_f1):.4f}")

Val Accuracy: max=0.7446, min=0.5822, avg=0.6990
Val F1:       max=0.6529, min=0.2546, avg=0.5935


In [ ]:
results = gru_model.evaluate(x_test, y_test, verbose=0)

loss, acc, f1_macro = results
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {acc * 100:.2f}%")
print(f"Test F1 Macro: {f1_macro:.4f}")

### AM-BiGRU-CNN

In [113]:
# Input
inputs = Input(shape=(max_len, 768))

# 1. Attention Layer
attn_out = AttentionLayer()(inputs)

# 2. BiGRU
bigru = Bidirectional(GRU(128, return_sequences=True, recurrent_dropout=0.1))(attn_out)
drop = Dropout(0.3)(bigru)


# 3. CNN
conv = Conv1D(filters=64, kernel_size=3, activation="gelu", padding="same")(bigru)
pool = GlobalMaxPooling1D()(conv)
drop = Dropout(0.3)(pool)

# 4. Dense
dense = Dense(32, activation="swish")(drop)
drop = Dropout(0.3)(dense)
dense = Dense(64, activation="sigmoid")(pool)
drop = Dropout(0.4)(dense)
outputs = Dense(num_classes, activation="softmax")(drop)

cbgat_model = Model(inputs=inputs, outputs=outputs)
cbgat_model.compile(loss='categorical_crossentropy', 
                   optimizer=Adam(learning_rate=1e-4), 
                   metrics=['accuracy',
                            F1Score(average='macro', name='f1_macro')]
                            )
cbgat_model.summary()


Model: "functional_31"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_31 (InputLayer)     │ (None, 95, 768)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ attention_layer_1               │ (None, 95, 768)        │           863 │
│ (AttentionLayer)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_20                │ (None, 95, 256)        │       689,664 │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_13 (Conv1D)              │ (None, 95, 64)         │        49,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_2          │ (None, 64)             │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_94 (Dense)                │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_99 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_95 (Dense)                │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 744,293 (2.84 MB)

 Trainable params: 744,293 (2.84 MB)

 Non-trainable params: 0 (0.00 B)

In [114]:
cbgat_history = cbgat_model.fit(x_train, y_train, epochs=200, batch_size = 32, validation_split=0.2, callbacks=callbacks)

Epoch 1/200
64/64 ━━━━━━━━━━━━━━━━━━━━ 15s 202ms/step - accuracy: 0.1568 - f1_macro: 0.1502 - loss: 1.9456 - val_accuracy: 0.6000 - val_f1_macro: 0.1253 - val_loss: 1.7180 - learning_rate: 1.0000e-04
Epoch 2/200
64/64 ━━━━━━━━━━━━━━━━━━━━ 16s 247ms/step - accuracy: 0.2114 - f1_macro: 0.1731 - loss: 1.8601 - val_accuracy: 0.6000 - val_f1_macro: 0.1250 - val_loss: 1.4028 - learning_rate: 1.0000e-04
Epoch 3/200
64/64 ━━━━━━━━━━━━━━━━━━━━ 19s 294ms/step - accuracy: 0.2985 - f1_macro: 0.1577 - loss: 1.7957 - val_accuracy: 0.6000 - val_f1_macro: 0.1250 - val_loss: 1.4162 - learning_rate: 1.0000e-04
Epoch 4/200
64/64 ━━━━━━━━━━━━━━━━━━━━ 19s 295ms/step - accuracy: 0.3078 - f1_macro: 0.1489 - loss: 1.7506 - val_accuracy: 0.6000 - val_f1_macro: 0.1250 - val_loss: 1.4148 - learning_rate: 1.0000e-04
Epoch 5/200
64/64 ━━━━━━━━━━━━━━━━━━━━ 19s 293ms/step - accuracy: 0.2958 - f1_macro: 0.1495 - loss: 1.7820 - val_accuracy: 0.6000 - val_f1_macro: 0.1250 - val_loss: 1.3834 - learning_rate: 1.0000e-04


In [128]:
history_dict = cbgat_history.history

val_acc = history_dict['val_accuracy'][-10:]
val_f1 = history_dict['val_f1_macro'][-10:]

print(f"Val Accuracy: max={np.max(val_acc):.4f}, min={np.min(val_acc):.4f}, avg={np.mean(val_acc):.4f}")
print(f"Val F1:       max={np.max(val_f1):.4f}, min={np.min(val_f1):.4f}, avg={np.mean(val_f1):.4f}")


Val Accuracy: max=0.7525, min=0.7149, avg=0.7356
Val F1:       max=0.6713, min=0.6409, avg=0.6577


In [ ]:
results = cbgat_model.evaluate(x_test, y_test, verbose=0)

loss, acc, f1_macro = results
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {acc * 100:.2f}%")
print(f"Test F1 Macro: {f1_macro:.4f}")

Test Loss: 1.6739
Test Accuracy: 42.33%
Test F1 Macro: 0.4215
